# Museum Classifier — Semi-supervised: Decision Tree + Label Propagation / Self-Training

**Preprocessing pipeline:**
- **Pipeline A** — ResNet18 (pretrained CNN feature extractor) → 512-d vectors

**Semi-supervised strategy (per config):**
1. Split training images into labeled (small ratio) + unlabeled
2. Label Propagation / LabelSpreading / Self-Training assigns pseudo-labels to unlabeled
3. Decision Tree retrained on labeled + pseudo-labeled combined
4. Compare against labeled-only DT baseline — evaluated on the validation set

**Dataset structure on Google Drive:**
```
MyDrive/appliedAI/
├── training/
│   ├── museum-indoor/
│   └── museum-outdoor/
├── museum_validation/
│   ├── museum-indoor/
│   └── museum-outdoor/
└── test/
```

In [ ]:
# Install missing packages (run once)
!pip install -q scikit-image opencv-python-headless torch torchvision

## Environment Setup

Run this notebook **locally** or on **Google Colab** — the cell below detects the environment automatically.

- **Local**: uses `~/Documents/Prog/AppliedAI` as the base folder
- **Colab**: mounts Google Drive and uses `MyDrive/appliedAI` — upload your dataset there first

In [ ]:
import os

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/appliedAI')  # ← adjust Drive path if needed
else:
    BASE_DIR = Path.home() / 'Documents' / 'Prog' / 'AppliedAI'  # local path

print(f'Running in: {"Google Colab" if _in_colab() else "local"} environment')
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
import os, sys, time, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset

from sklearn.tree import DecisionTreeClassifier
from sklearn.semi_supervised import LabelPropagation, LabelSpreading, SelfTrainingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from skimage.feature import hog, local_binary_pattern
from skimage import color as skcolor
import cv2

%matplotlib inline
warnings.filterwarnings('ignore')
import joblib

## Configuration

Adjust `BASE_DIR` to point to your dataset folder inside Google Drive if needed.

In [ ]:
TRAIN_DIR  = BASE_DIR / 'training'
VAL_DIR    = BASE_DIR / 'museum_validation'
TEST_DIR   = BASE_DIR / 'test'
OUTPUT_DIR = BASE_DIR / 'outputs_semisupervised'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['museum-indoor', 'museum-outdoor']   # must match folder names
IMG_SIZE     = 224
BATCH_SIZE   = 32
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42

print(f'Training dir    : {TRAIN_DIR}')
print(f'Validation dir  : {VAL_DIR}')
print(f'Test dir        : {TEST_DIR}')
print(f'Device          : {DEVICE}')
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint file paths — shared with supervised notebook where possible
CKPT_RESNET_TRAIN  = CHECKPOINT_DIR / 'sup_resnet_train.npz'   # reused if supervised ran first
CKPT_RESNET_VAL    = CHECKPOINT_DIR / 'sup_resnet_val.npz'
CKPT_MODELS_SEMI_R = CHECKPOINT_DIR / 'semi_models_resnet.joblib'


## Dataset Loader

In [ ]:
class MuseumDataset(Dataset):
    """Loads labeled images from a root folder containing one sub-folder per class."""
    def __init__(self, root: Path, classes: list, transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(classes):
            d = root / cls
            if not d.exists():
                print(f'[WARN] Folder not found: {d}'); continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
                for p in d.glob(ext):
                    self.samples.append((p, idx))
        counts = {cls: sum(1 for _, l in self.samples if l == i)
                  for i, cls in enumerate(classes)}
        print(f'  {root.name}: {len(self.samples)} images — ' +
              ', '.join(f'{cls}={n}' for cls, n in counts.items()))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

In [ ]:
print('Loading datasets...')
train_dataset = MuseumDataset(TRAIN_DIR, CLASSES)
val_dataset   = MuseumDataset(VAL_DIR,   CLASSES)

if len(train_dataset) == 0:
    raise RuntimeError('No training images found — check TRAIN_DIR.')
if len(val_dataset) == 0:
    raise RuntimeError('No validation images found — check VAL_DIR.')

## Step 1-A — Image Preprocessing: ResNet18 (512-d)

Pretrained ResNet18 with the classification head replaced by `nn.Identity()`.
Each image → 512-dimensional embedding. Run separately on training and validation sets.

In [ ]:
def _load_npz(path):
    d = np.load(path)
    return d['X'], d['y']

def _save_npz(path, X, y):
    np.savez_compressed(path, X=X, y=y)
    print(f'  [CKPT] Saved → {path}')

def extract_resnet_features(dataset, ckpt_path=None):
    if ckpt_path and ckpt_path.exists():
        print(f'  Loading from checkpoint: {ckpt_path.name}')
        X, y = _load_npz(ckpt_path)
        print(f'    shape={X.shape}')
        return X, y
    resnet_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    dataset.transform = resnet_tf
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone.eval().to(DEVICE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    feats_l, labels_l = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            feats_l.append(backbone(imgs.to(DEVICE)).cpu().numpy())
            labels_l.append(lbls.numpy() if isinstance(lbls, torch.Tensor) else np.array(lbls))
    dataset.transform = None
    X = np.concatenate(feats_l)
    y = np.concatenate(labels_l)
    print(f'    shape={X.shape}')
    if ckpt_path:
        _save_npz(ckpt_path, X, y)
    return X, y

print('[STEP 1-A] Extracting ResNet18 features...')
print('  Training set:')
X_resnet_train, y_train = extract_resnet_features(train_dataset, CKPT_RESNET_TRAIN)
print('  Validation set:')
X_resnet_val,   y_val   = extract_resnet_features(val_dataset,   CKPT_RESNET_VAL)

## Semi-supervised Split Helper

Stratified split applied to the **training set only**: a given ratio per class receives real
labels; the rest are marked `-1` (unlabeled) for graph-based propagation methods.
The validation set is always fully labeled and used only for final evaluation.

In [ ]:
def make_semisup_split(X, y, labeled_ratio, rng_seed):
    """Stratified split: labeled_ratio% per class gets a real label, rest = -1."""
    rng = np.random.RandomState(rng_seed)
    labeled_idx = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        n_lab   = max(1, int(len(cls_idx) * labeled_ratio))
        labeled_idx.extend(rng.choice(cls_idx, n_lab, replace=False).tolist())
    labeled_idx   = np.array(labeled_idx)
    unlabeled_idx = np.setdiff1d(np.arange(len(y)), labeled_idx)
    y_semi = np.full(len(y), -1, dtype=int)
    y_semi[labeled_idx] = y[labeled_idx]
    return labeled_idx, unlabeled_idx, y_semi

## Step 2 — Semi-supervised Configurations (× 5)

Five configurations varying labeled ratio (10–30%), propagation method (LP / LS / Self-Training),
and Decision Tree depth.

In [ ]:
# 5 semi-supervised configurations varying:
# labeled_ratio (10–30%), propagation method (LP/LS/ST),
# dt_max_depth (≤2 for computational feasibility), and lp_kernel.
# (config_name, labeled_ratio, method, dt_max_depth, lp_kernel)
SEMISUP_CONFIGS = [
    # 10% labeled, Label Propagation rbf, shallow DT
    ('LP_rbf_10pct_DT1',         0.10, 'LP', 1,    'rbf'),
    # 20% labeled, Label Propagation knn — explores kernel effect
    ('LP_knn_20pct_DT2',         0.20, 'LP', 2,    'knn'),
    # 30% labeled, Label Propagation rbf, deeper DT — more labeled data
    ('LP_rbf_30pct_DT2',         0.30, 'LP', 2,    'rbf'),
    # 20% labeled, Self-Training — no graph, DT base, threshold=0.75
    ('SelfTrain_20pct_DT2',      0.20, 'ST', 2,    None),
    # 30% labeled, LabelSpreading rbf — softer clamping (alpha=0.2)
    ('LabelSpreading_30pct_DT2', 0.30, 'LS', 2,    'rbf'),
]

In [ ]:
def run_semisup_pipeline(X_train: np.ndarray, y_train: np.ndarray,
                          X_val:   np.ndarray, y_val:   np.ndarray,
                          pipe_label: str, ckpt_path=None) -> tuple:
    """Scale → PCA (for LP/LS) → semi-sup propagation → DT retrain → eval on val.
    Checkpoint: pass ckpt_path to save/load results. Delete the file to retrain.
    """
    if ckpt_path and ckpt_path.exists():
        print(f'\n[STEP 2] Loading semi-sup models from checkpoint: {ckpt_path.name} …')
        payload = joblib.load(ckpt_path)
        results, scaler = payload['results'], payload['scaler']
        for name, r in results.items():
            print(f'  [LOADED] {name}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  ΔF1={r["f1"]-r["f1_base"]:+.4f}')
        return results, scaler
    scaler      = StandardScaler()
    X_train_sc  = scaler.fit_transform(X_train)
    X_val_sc    = scaler.transform(X_val)

    # PCA reduces cost of graph-based methods (O(N^2 x d))
    pca         = PCA(n_components=min(64, X_train_sc.shape[1]-1), random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_sc)
    print(f'  PCA 64-d: explained variance = {pca.explained_variance_ratio_.sum():.3f}')

    results = {}
    print(f'\n{"-"*64}')
    print(f'  {pipe_label}')
    print(f'  features={X_train.shape[1]}  train={len(X_train)}  val={len(X_val)}')
    print(f'{"-"*64}')

    for (name, labeled_ratio, method, dt_depth, lp_kernel) in SEMISUP_CONFIGS:
        print(f'\n  [CONFIG] {name}  (labeled={labeled_ratio:.0%}  method={method})')
        t0 = time.time()

        labeled_idx, unlabeled_idx, y_semi = make_semisup_split(
            X_train_pca, y_train, labeled_ratio, RANDOM_STATE)
        print(f'    Labeled={len(labeled_idx)}  Unlabeled={len(unlabeled_idx)}')

        # Semi-supervised propagation on PCA-reduced training features
        if method == 'LP':
            lp = LabelPropagation(kernel=lp_kernel, n_neighbors=7, max_iter=1000, n_jobs=-1)
            lp.fit(X_train_pca, y_semi)
            pseudo_all = lp.transduction_
        elif method == 'LS':
            ls = LabelSpreading(kernel=lp_kernel, alpha=0.2, max_iter=1000, n_jobs=-1)
            ls.fit(X_train_pca, y_semi)
            pseudo_all = ls.transduction_
        else:  # Self-Training
            base = DecisionTreeClassifier(max_depth=dt_depth, random_state=RANDOM_STATE)
            st   = SelfTrainingClassifier(base_estimator=base, threshold=0.75,
                                           criterion='threshold', max_iter=10, verbose=False)
            st.fit(X_train_pca, y_semi)
            pseudo_all = st.transduction_

        pseudo_unlab = pseudo_all[unlabeled_idx]
        pseudo_acc   = accuracy_score(y_train[unlabeled_idx], pseudo_unlab)
        print(f'    Pseudo-label accuracy: {pseudo_acc:.4f}')

        # Retrain DT on labeled + pseudo-labeled (full feature space, not PCA)
        X_combined = np.vstack([X_train_sc[labeled_idx], X_train_sc[unlabeled_idx]])
        y_combined = np.concatenate([y_train[labeled_idx], pseudo_unlab])
        dt = DecisionTreeClassifier(max_depth=dt_depth, min_samples_leaf=3,
                                     random_state=RANDOM_STATE)
        dt.fit(X_combined, y_combined)

        # Labeled-only baseline (evaluated on val)
        dt_base = DecisionTreeClassifier(max_depth=dt_depth, random_state=RANDOM_STATE)
        dt_base.fit(X_train_sc[labeled_idx], y_train[labeled_idx])
        f1_base = f1_score(y_val, dt_base.predict(X_val_sc), zero_division=0)

        # Evaluate semi-sup DT on validation set
        y_pred  = dt.predict(X_val_sc)
        y_proba = dt.predict_proba(X_val_sc)[:, 1]
        acc     = accuracy_score(y_val, y_pred)
        f1      = f1_score(y_val, y_pred, zero_division=0)
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        roc_auc = auc(fpr, tpr)
        elapsed = time.time() - t0

        results[name] = dict(
            dt=dt, scaler=scaler,
            acc=acc, f1=f1, f1_base=f1_base, roc_auc=roc_auc,
            fpr=fpr, tpr=tpr, cm=confusion_matrix(y_val, y_pred),
            y_pred=y_pred, y_proba=y_proba, y_test=y_val,
            pseudo_acc=pseudo_acc, n_lab=len(labeled_idx), n_unlab=len(unlabeled_idx),
            labeled_ratio=labeled_ratio, time=elapsed,
            report=classification_report(y_val, y_pred, target_names=CLASSES),
        )
        print(f'    Acc={acc:.4f}  F1={f1:.4f}  F1_base={f1_base:.4f}  '
              f'ΔF1={f1-f1_base:+.4f}  AUC={roc_auc:.4f}  t={elapsed:.1f}s')

    if ckpt_path:
        joblib.dump({'results': results, 'scaler': scaler}, ckpt_path)
        print(f'  [CKPT] Saved → {ckpt_path}')
    return results, scaler

In [ ]:
# ── PIPELINE A ─ Run (or load from checkpoint) ──────────────────────────────
print('=' * 64)
print('  Running Pipeline A — ResNet18 (semi-supervised)')
print('=' * 64)
# ResNet features: 512-d  |  saves to CKPT_MODELS_SEMI_R
res_r, sc_r = run_semisup_pipeline(
    X_resnet_train, y_train, X_resnet_val, y_val,
    'Pipeline A — ResNet18', CKPT_MODELS_SEMI_R
)
print('\n[Pipeline A done — checkpoint saved. You can stop here and resume later.]')


In [ ]:
# ── COMBINE RESULTS ────────────────────────────────────────────────────────
all_results = {'resnet': res_r}
scalers     = {'resnet': sc_r}
print('Results ready — generating report.')
for cn, r in all_results['resnet'].items():
    f1_delta = r['f1'] - r['f1_base']
    print(f'  {cn:35s}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  ΔF1={f1_delta:+.4f}')


## Step 3 — Report

Comprehensive dashboard: accuracy/F1/AUC bars, F1 semi vs baseline, ΔF1 heatmap, ROC curves,
pseudo-label accuracy, best confusion matrices, labeled vs pseudo-labeled counts,
ratio→ΔF1 scatter, training times, cross-pipeline ΔF1, and full summary table.

In [ ]:
PIPE_META = {
    'resnet': {'label': 'Pipeline A — ResNet18 (512-d)', 'color': '#4FC3F7'},
}

def build_report(all_results: dict):
    cfg_names = [c[0] for c in SEMISUP_CONFIGS]
    n         = len(cfg_names)
    palette   = sns.color_palette('Set2', n)
    color     = PIPE_META['resnet']['color']

    fig = plt.figure(figsize=(20, 30))
    fig.patch.set_facecolor('#0d0f1a')
    gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.62, wspace=0.40)
    tkw  = dict(color='white', fontsize=10, fontweight='bold', pad=8)
    axbg = '#161929'

    def sa(ax):
        ax.set_facecolor(axbg); ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
        for sp in ax.spines.values(): sp.set_color('#2a2d3e')

    xlbls = [c.replace('_', '\n') for c in cfg_names]
    x, w  = np.arange(n), 0.6

    # Row 0: Accuracy / F1 / AUC bars
    for col, (mkey, mtitle) in enumerate([('acc','Accuracy'),('f1','F1 Score (Semi)'),('roc_auc','ROC-AUC')]):
        ax = fig.add_subplot(gs[0, col]); sa(ax)
        vals = [all_results['resnet'][c][mkey] for c in cfg_names]
        bars = ax.bar(x, vals, w, color=color, edgecolor='white', linewidth=0.4, alpha=0.88)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.012,
                    f'{v:.2f}', ha='center', color='white', fontsize=6.5)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
        ax.set_ylim(0, 1.15); ax.set_title(mtitle, **tkw)

    # Row 1: F1 semi vs baseline (spans 2 cols) + ΔF1 heatmap
    ax_f1 = fig.add_subplot(gs[1, 0:2]); sa(ax_f1)
    f1s  = [all_results['resnet'][c]['f1']      for c in cfg_names]
    f1bs = [all_results['resnet'][c]['f1_base'] for c in cfg_names]
    ax_f1.bar(x - w/4, f1s,  w/2, color=color, label='Semi-sup DT',  alpha=0.90, edgecolor='white', lw=0.4)
    ax_f1.bar(x + w/4, f1bs, w/2, color=color, label='Labeled-only', alpha=0.35, edgecolor='white', lw=0.4)
    ax_f1.set_xticks(x); ax_f1.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_f1.set_ylim(0, 1.12)
    ax_f1.set_title('F1: Semi-supervised vs Labeled-only Baseline', **tkw)
    ax_f1.legend(fontsize=8, facecolor='#0d0f1a', labelcolor='white')

    ax_heat = fig.add_subplot(gs[1, 2]); sa(ax_heat)
    delta_vals = np.array([[all_results['resnet'][c]['f1'] - all_results['resnet'][c]['f1_base']
                            for c in cfg_names]])
    lim = max(abs(delta_vals.min()), abs(delta_vals.max())) + 0.01
    im = ax_heat.imshow(delta_vals, cmap='RdYlGn', aspect='auto', vmin=-lim, vmax=lim)
    ax_heat.set_xticks(range(n)); ax_heat.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_heat.set_yticks([0]); ax_heat.set_yticklabels(['ResNet18'], color='#ccc', fontsize=8)
    ax_heat.set_title('ΔF1 Heatmap (Semi − Baseline)\nGreen = semi helps', **tkw)
    for j in range(n):
        ax_heat.text(j, 0, f'{delta_vals[0,j]:+.3f}', ha='center', va='center',
                     color='black', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)

    # Row 2: ROC curves (spans 2 cols) + pseudo-label accuracy
    ax_roc = fig.add_subplot(gs[2, 0:2]); sa(ax_roc)
    ax_roc.plot([0,1],[0,1],'w--',lw=1,alpha=0.35)
    for i, cn in enumerate(cfg_names):
        r = all_results['resnet'][cn]
        ax_roc.plot(r['fpr'], r['tpr'], color=palette[i], lw=1.8,
                    label=f'{cn} ({r["roc_auc"]:.3f})')
    ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title('ROC Curves — Pipeline A: ResNet18', **tkw)
    ax_roc.legend(fontsize=6, facecolor='#0d0f1a', labelcolor='white')

    ax_pseudo = fig.add_subplot(gs[2, 2]); sa(ax_pseudo)
    pa = [all_results['resnet'][c]['pseudo_acc'] for c in cfg_names]
    bars_p = ax_pseudo.bar(x, pa, w, color=color, edgecolor='white', linewidth=0.4, alpha=0.88)
    for bar, v in zip(bars_p, pa):
        ax_pseudo.text(bar.get_x()+bar.get_width()/2, v+0.01,
                       f'{v:.3f}', ha='center', color='white', fontsize=7)
    ax_pseudo.set_xticks(x); ax_pseudo.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_pseudo.set_ylim(0, 1.1)
    ax_pseudo.set_title('Pseudo-label Accuracy\n(quality of unlabeled propagation)', **tkw)

    # Row 3: Best confusion matrix + labeled/pseudo counts + ratio vs ΔF1
    best_cn = max(cfg_names, key=lambda c: all_results['resnet'][c]['f1'])
    ax_cm = fig.add_subplot(gs[3, 0]); sa(ax_cm)
    ConfusionMatrixDisplay(all_results['resnet'][best_cn]['cm'],
                           display_labels=CLASSES).plot(ax=ax_cm, colorbar=False, cmap='Blues')
    ax_cm.set_title(f'Best CM — ResNet18\n{best_cn}', **tkw)
    ax_cm.xaxis.label.set_color('#ccc'); ax_cm.yaxis.label.set_color('#ccc')

    ax_stk = fig.add_subplot(gs[3, 1]); sa(ax_stk)
    n_labs   = [all_results['resnet'][c]['n_lab']   for c in cfg_names]
    n_unlabs = [all_results['resnet'][c]['n_unlab'] for c in cfg_names]
    ax_stk.bar(range(n), n_labs,   color='#4CAF50', label='Labeled',         alpha=0.9)
    ax_stk.bar(range(n), n_unlabs, bottom=n_labs, color='#2196F3', label='Pseudo-labeled', alpha=0.65)
    ax_stk.set_xticks(range(n)); ax_stk.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_stk.set_title('Labeled vs Pseudo-labeled Counts', **tkw)
    ax_stk.legend(fontsize=8, facecolor='#0d0f1a', labelcolor='white')

    ax_sc = fig.add_subplot(gs[3, 2]); sa(ax_sc)
    ratios = [all_results['resnet'][c]['labeled_ratio'] for c in cfg_names]
    deltas = [all_results['resnet'][c]['f1'] - all_results['resnet'][c]['f1_base'] for c in cfg_names]
    ax_sc.scatter(ratios, deltas, c=color, s=90, edgecolors='white', linewidth=0.5, zorder=3)
    for i, cn in enumerate(cfg_names):
        ax_sc.annotate(cn.split('_')[0], (ratios[i], deltas[i]),
                       textcoords='offset points', xytext=(5, 3), color='#ccc', fontsize=6)
    ax_sc.axhline(0, color='white', lw=0.8, alpha=0.5, linestyle='--')
    ax_sc.set_xlabel('Labeled Ratio'); ax_sc.set_ylabel('ΔF1')
    ax_sc.set_title('Labeled Ratio → ΔF1 Gain', **tkw)

    # Row 4: Training time + summary table
    ax_time = fig.add_subplot(gs[4, 0]); sa(ax_time)
    times = [all_results['resnet'][c]['time'] for c in cfg_names]
    ax_time.bar(x, times, w, color=color, edgecolor='white', linewidth=0.4, alpha=0.88)
    ax_time.set_xticks(x); ax_time.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_time.set_ylabel('seconds'); ax_time.set_title('Training Time (s)', **tkw)

    ax_tbl = fig.add_subplot(gs[4, 1:]); sa(ax_tbl); ax_tbl.axis('off')
    col_labels = ['Config','Labeled%','Accuracy','F1 Semi','F1 Base','ΔF1','Pseudo-Acc','AUC','Time(s)']
    rows = []
    for cn in cfg_names:
        r = all_results['resnet'][cn]
        delta = r['f1'] - r['f1_base']
        rows.append([
            cn, f'{r["labeled_ratio"]:.0%}',
            f'{r["acc"]:.4f}', f'{r["f1"]:.4f}', f'{r["f1_base"]:.4f}',
            f'{delta:+.4f}', f'{r["pseudo_acc"]:.4f}',
            f'{r["roc_auc"]:.4f}', f'{r["time"]:.1f}',
        ])
    tbl = ax_tbl.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.90)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#2a2d3e')
        if row == 0:
            cell.set_facecolor('#1e2235'); cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#0d1828' if row%2 else '#091220')
            if col == 5:
                try:
                    v = float(rows[row-1][5])
                    cell.set_text_props(color='#4CAF50' if v >= 0 else '#EF5350', fontweight='bold')
                except: cell.set_text_props(color='#B3E5FC')
            else: cell.set_text_props(color='#B3E5FC')
    ax_tbl.set_title('Results Summary — Pipeline A: ResNet18', **tkw)

    fig.text(0.5, 0.988, 'Museum Classifier — Semi-supervised Decision Tree',
             ha='center', va='top', color='white', fontsize=17, fontweight='bold')
    fig.text(0.5, 0.975, 'Step 3 Report  |  Pipeline A: ResNet18 (512-d)',
             ha='center', va='top', color='#aaa', fontsize=10)

    out = OUTPUT_DIR / 'report_semisupervised.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'\n[REPORT] Saved → {out}')
    return out


In [ ]:
report_path = build_report(all_results)

## Best Model Classification Reports

In [ ]:
for pk in all_results:
    best = max(all_results[pk], key=lambda c: all_results[pk][c]['f1'])
    print(f'[BEST — {PIPE_META[pk]["label"]}]  {best}')
    print(all_results[pk][best]['report'])
    print()

## Test Prediction

Uses the best model (Pipeline A — ResNet18) to predict unlabeled images in `TEST_DIR`.
Saves a CSV with filename, predicted class, confidence, pipeline, and config.


In [ ]:
def predict_test(all_results: dict, scalers: dict):
    best_pk, best_cn, best_f1 = None, None, -1
    for pk in all_results:
        for cn in all_results[pk]:
            if all_results[pk][cn]['f1'] > best_f1:
                best_pk, best_cn, best_f1 = pk, cn, all_results[pk][cn]['f1']

    print(f'Best model: {PIPE_META[best_pk]["label"]} / {best_cn}  (F1={best_f1:.4f})')

    test_paths = []
    if TEST_DIR.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
            test_paths.extend(TEST_DIR.glob(ext))
    if not test_paths:
        print('[WARN] No test images found in TEST_DIR.'); return

    dt, scaler = all_results[best_pk][best_cn]['dt'], scalers[best_pk]

    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity(); backbone.eval().to(DEVICE)
    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    feats = []
    with torch.no_grad():
        for p in test_paths:
            img = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
            feats.append(backbone(img).cpu().numpy()[0])

    X_test = scaler.transform(np.array(feats))
    preds  = dt.predict(X_test)
    probas = dt.predict_proba(X_test)[:, 1]

    out_csv = OUTPUT_DIR / 'predictions_semisupervised.csv'
    with open(out_csv, 'w') as f:
        f.write('filename,prediction,confidence_outdoor,pipeline,config\n')
        for p, pred, prob in zip(test_paths, preds, probas):
            f.write(f'{p.name},{CLASSES[pred]},{prob:.4f},{best_pk},{best_cn}\n')
    print(f'Predictions saved → {out_csv}')
    print(f'[DONE] All outputs in: {OUTPUT_DIR}')

predict_test(all_results, scalers)
